# DocuVision AI — Training Notebook (run on Colab, free T4 GPU)

Runtime > Change runtime type > **T4 GPU** before running this.

In [ ]:
!nvidia-smi

In [2]:
from typing_extensions import Doc
%cd /content
!rm -rf DocuVisionAI
!git clone https://github.com/fathimarfa/DocuVisionAI.git
%cd DocuVisionAI

/content
Cloning into 'DocuVisionAI'...
remote: Enumerating objects: 78, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 78 (delta 31), reused 60 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (78/78), 25.35 KiB | 25.35 MiB/s, done.
Resolving deltas: 100% (31/31), done.
/content/DocuVisionAI


In [3]:
!cat requirements.txt

transformers==4.46.3
sentencepiece>=0.1.99
datasets>=2.19.0
Pillow>=10.1.0
jiwer==3.0.0
accelerate>=0.24.0
tensorboard>=2.15.0
PyYAML>=6.0.1
tqdm>=4.66.1
pandas>=2.1.3
numpy>=1.26.2
scikit-learn>=1.3.2
pytest>=7.4.3

In [4]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 129.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 118.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [5]:
# 2. Download + split the dataset
!python data/download_dataset.py --config configs/trocr_base.yaml

2026-07-21 16:44:31,749 [INFO] Downloading dataset: Teklia/IAM-line
2026-07-21 16:44:34,991 [INFO] Available splits: ['train', 'validation', 'test']
Generating train split: 100% 6482/6482 [00:00<00:00, 16469.66 examples/s]
Generating validation split: 100% 976/976 [00:00<00:00, 14889.65 examples/s]
Generating test split: 100% 2915/2915 [00:00<00:00, 14927.75 examples/s]
2026-07-21 16:45:03,255 [INFO] Wrote train (500 rows) -> data/processed/train.csv
2026-07-21 16:45:10,120 [INFO] Wrote val (97 rows) -> data/processed/val.csv
2026-07-21 16:45:26,727 [INFO] Wrote test (291 rows) -> data/processed/test.csv


In [6]:
# 3. Fine-tune TrOCR (mixed precision + gradient accumulation configured in YAML)
!python -m src.train --config configs/trocr_base.yaml

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
2026-07-21 16:51:09.779157: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
vocab.json: 899kB [00:00, 92.4MB/s]
merges.txt: 456kB [00:00, 157MB/s]
model.safetensors: 100% 1.33G/1.33G [00:13<00:00, 103MB/s] 
Config of the encoder: <class 'transformers.models.vit.modeling_vit.ViTModel'> is overwritten by shared encoder config: ViTConfig {
  "attention_probs_dropout_prob": 0.0,
  "encoder_stride": 16,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "image_size": 384,
  "ini

In [ ]:
# 4. Evaluate: fine-tuned vs. base model zero-shot (CER / WER report)
!python src/evaluate.py --checkpoint weights/checkpoint-best --base-model microsoft/trocr-base-handwritten

In [ ]:
# 5. Zip the best checkpoint and download it locally (for use in VS Code / app.py)
!zip -r checkpoint-best.zip weights/checkpoint-best
from google.colab import files
files.download('checkpoint-best.zip')

In [ ]:
# 6. (Optional) push the trained checkpoint + results back to your repo
# !git config --global user.email "you@example.com"
# !git config --global user.name "Your Name"
# !git add weights/checkpoint-best
# !git commit -m "Add trained checkpoint"
# !git push